# 00 — Tìm hiểu dữ liệu Wine Reviews

Notebook này chỉ đọc và kiểm tra dữ liệu gốc. Không có thao tác làm sạch, ghi đè hoặc xuất dữ liệu mới.

## 1. Nạp thư viện và xác định đường dẫn

Đường dẫn dưới đây hoạt động khi chạy notebook từ thư mục gốc project hoặc từ thư mục `notebooks`.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'winemag-data-130k-v2.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy dataset tại: {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
print(f'Dataset: {DATA_PATH}')
print(f'Đã đọc thành công: {df.shape[0]:,} dòng × {df.shape[1]} cột')

Dataset: C:\Users\pnvto\Data Science\wine-quality-prediction\data\raw\winemag-data-130k-v2.csv
Đã đọc thành công: 129,971 dòng × 14 cột


## 2. Kích thước, tên cột và 10 dòng đầu

In [2]:
print('Kích thước:', df.shape)
print('Tên cột:', df.columns.tolist())
print('\n10 dòng đầu:')
print(df.head(10).to_string(index=False))

Kích thước: (129971, 14)
Tên cột: ['Unnamed: 0', 'country', 'description', 'designation', 'points', 'price', 'province', 'region_1', 'region_2', 'taster_name', 'taster_twitter_handle', 'title', 'variety', 'winery']

10 dòng đầu:
 Unnamed: 0  country                                                                                                                                                                                                                                                           description                        designation  points  price          province            region_1          region_2        taster_name taster_twitter_handle                                                                               title            variety              winery
          0    Italy                                                                                          Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering

## 3. Kiểu dữ liệu và thông tin tổng quát

In [3]:
df.info()
print('\nKiểu dữ liệu theo cột:')
print(df.dtypes.to_string())

<class 'pandas.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Unnamed: 0             129971 non-null  int64  
 1   country                129908 non-null  str    
 2   description            129971 non-null  str    
 3   designation            92506 non-null   str    
 4   points                 129971 non-null  int64  
 5   price                  120975 non-null  float64
 6   province               129908 non-null  str    
 7   region_1               108724 non-null  str    
 8   region_2               50511 non-null   str    
 9   taster_name            103727 non-null  str    
 10  taster_twitter_handle  98758 non-null   str    
 11  title                  129971 non-null  str    
 12  variety                129970 non-null  str    
 13  winery                 129971 non-null  str    
dtypes: float64(1), int64(2), str(11)
memory usage: 

## 4. Giá trị thiếu và số giá trị phân biệt

In [4]:
column_profile = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'unique_non_null': df.nunique(dropna=True),
})
print(column_profile.sort_values('missing_count', ascending=False).to_string())

                         dtype  missing_count  missing_pct  unique_non_null
region_2                   str          79460        61.14               17
designation                str          37465        28.83            37979
taster_twitter_handle      str          31213        24.02               15
taster_name                str          26244        20.19               19
region_1                   str          21247        16.35             1229
price                  float64           8996         6.92              390
province                   str             63         0.05              425
country                    str             63         0.05               43
variety                    str              1         0.00              707
Unnamed: 0               int64              0         0.00           129971
points                   int64              0         0.00               21
description                str              0         0.00           119955
title       

## 5. Thống kê mô tả các cột số

In [5]:
print(df.describe().round(2).to_string())

       Unnamed: 0     points      price
count   129971.00  129971.00  120975.00
mean     64985.00      88.45      35.36
std      37519.54       3.04      41.02
min          0.00      80.00       4.00
25%      32492.50      86.00      17.00
50%      64985.00      88.00      25.00
75%      97477.50      91.00      42.00
max     129970.00     100.00    3300.00


## 6. Kiểm tra bản ghi trùng

Cột `Unnamed: 0` là chỉ mục được lưu kèm trong CSV. Vì cột này là duy nhất, cần kiểm tra trùng cả khi có và khi loại cột đó khỏi phép so sánh. Notebook chỉ đếm, chưa xóa bản ghi nào.

In [6]:
index_column = 'Unnamed: 0'
duplicate_all_columns = int(df.duplicated().sum())
content_columns = [column for column in df.columns if column != index_column]
duplicate_without_index = int(df.duplicated(subset=content_columns).sum())

print(f'Trùng khi so sánh toàn bộ 14 cột: {duplicate_all_columns:,}')
print(f'Trùng khi không dùng {index_column}: {duplicate_without_index:,}')
print(f'{index_column} có duy nhất không: {df[index_column].is_unique}')

Trùng khi so sánh toàn bộ 14 cột: 0
Trùng khi không dùng Unnamed: 0: 9,983
Unnamed: 0 có duy nhất không: True


## 7. Dấu hiệu cần kiểm tra thêm ở dữ liệu văn bản

Các phép kiểm tra dưới đây chỉ tìm khoảng trắng ở đầu/cuối và các cách viết chỉ khác chữ hoa/chữ thường. Đây là dấu hiệu audit, chưa phải quyết định làm sạch.

In [7]:
text_columns = df.select_dtypes(include=['object', 'string']).columns
whitespace_counts = {}
for column in text_columns:
    values = df[column].dropna().astype(str)
    whitespace_counts[column] = int((values != values.str.strip()).sum())

case_variant_counts = {}
for column in ['country', 'province', 'region_1', 'region_2', 'variety', 'winery']:
    values = df[column].dropna().astype(str)
    pairs = pd.DataFrame({'normalized': values.str.strip().str.casefold(), 'original': values})
    variants = pairs.groupby('normalized')['original'].nunique()
    case_variant_counts[column] = int((variants > 1).sum())

print('Số giá trị có khoảng trắng đầu/cuối:')
print(pd.Series(whitespace_counts).sort_values(ascending=False).to_string())
print('\nSố nhóm có biến thể chữ hoa/thường sau khi chuẩn hóa:')
print(pd.Series(case_variant_counts).sort_values(ascending=False).to_string())

Số giá trị có khoảng trắng đầu/cuối:
taster_twitter_handle    9532
winery                     17
title                      14
designation                 5
region_1                    4
variety                     4
description                 1
country                     0
taster_name                 0
province                    0
region_2                    0

Số nhóm có biến thể chữ hoa/thường sau khi chuẩn hóa:
winery      27
country      0
province     0
region_1     0
region_2     0
variety      0


## 8. Kết luận audit ban đầu

- Dataset có **129.971 dòng và 14 cột**.
- Cột `Unnamed: 0` là chỉ mục CSV, có giá trị từ 0 đến 129.970 và không trùng. Nhóm cần thống nhất trước khi bỏ cột.
- Có dữ liệu thiếu ở 9 cột; `region_2` thiếu nhiều nhất.
- Không có dòng trùng khi tính cả chỉ mục, nhưng có **9.983** dòng trùng nếu bỏ chỉ mục khỏi phép so sánh.
- `price` có giá trị tối đa 3.300 USD, cao hơn nhiều so với trung vị 25 USD; đây là ứng viên cần phân tích outlier, chưa kết luận là lỗi.
- Có dấu hiệu khoảng trắng ở một số cột văn bản và 27 nhóm tên `winery` chỉ khác cách viết hoa/thường.
- Mọi quyết định xử lý được để lại cho nhóm trong `docs/cleaning_log.csv`.